# LeetCode 910 — Smallest Range II bằng Python

Link đề: https://leetcode.com/problems/smallest-range-ii/ · Độ khó: **Medium** · Chủ đề: Array (mảng), Math (toán), Greedy (tham lam), Sorting (sắp xếp)

Skeleton LeetCode:

```python
class Solution:
    def smallestRangeII(self, nums: List[int], k: int) -> int:
        ...
```

Ghi chú: **mỗi** phần tử phải được **cộng `k` hoặc trừ `k`** (đúng một trong hai, không được giữ nguyên), rồi ta tính điểm = `max - min`. Sau khi **sắp xếp**, chiến lược tối ưu luôn có dạng: **một tiền tố** của dãy được `+k` và **phần còn lại** được `-k` (số nhỏ tăng, số lớn giảm làm thu hẹp khoảng cách). Do đó chỉ cần thử **mọi điểm cắt**: `n` khả năng thay vì `2^n`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mỗi `nums[i]` biến thành `nums[i] + k` hoặc `nums[i] - k`; cần điểm `max - min` nhỏ nhất.
2. Nắm các công cụ Python cần thiết: `sort`, `min`/`max` nhiều đối số, `itertools.product` để liệt kê dấu `+/-`, tiền tố/hậu tố min/max, và so sánh mảng số nguyên.
3. Hiểu nguyên lý tham lam bằng **lập luận hoán đổi**: nếu một số nhỏ hơn bị `-k` trong khi một số lớn hơn được `+k` thì **đổi dấu** hai số đó không làm điểm tệ đi; do đó tồn tại cách tối ưu mà các số `+k` đứng trước các số `-k` (sau khi sắp xếp).
4. Đi qua 4 cách: thử `2^n` tổ hợp dấu (brute), thử mọi điểm cắt bằng quét `O(n)` mỗi điểm, thử mọi điểm cắt bằng công thức `O(1)` (cách nộp chính), và liệt kê **cận dưới** của đoạn kết quả; đối chiếu bằng oracle vét cạn.
5. Tránh các lỗi kinh điển: bỏ qua trường hợp "không cắt" (mọi số cùng dấu), không sắp xếp, tham lam từng phần tử, và nhầm với bài **Smallest Range I** (908) cho phép cộng một số tùy ý trong `[-k, k]`.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` và số nguyên `k`.

Với mỗi chỉ số `i`, hãy đổi `nums[i]` thành `nums[i] + k` **hoặc** `nums[i] - k`.

**Điểm** của `nums` là hiệu giữa phần tử lớn nhất và nhỏ nhất. Trả về **điểm nhỏ nhất** sau khi đã đổi mọi phần tử.

**Ví dụ 1**

- Input: `nums = [1]`, `k = 0`
- Output: `0`

**Ví dụ 2**

- Input: `nums = [0,10]`, `k = 2`
- Output: `6` (đổi thành `[2, 8]`)

**Ví dụ 3**

- Input: `nums = [1,3,6]`, `k = 3`
- Output: `3` (đổi thành `[4, 6, 3]`)

**Ràng buộc:**

- `1 <= nums.length <= 10^4`
- `0 <= nums[i] <= 10^4`
- `0 <= k <= 10^4`

In [1]:
from typing import List
from itertools import product
import random, time

examples = [
    ([1], 0, 0),
    ([0, 10], 2, 6),
    ([1, 3, 6], 3, 3),
]
for nums, k, expected in examples:
    print(f"nums={nums} k={k} kỳ vọng={expected}")

nums=[1] k=0 kỳ vọng=0
nums=[0, 10] k=2 kỳ vọng=6
nums=[1, 3, 6] k=3 kỳ vọng=3


## 3. Kiến thức nền

### 3.1. Mỗi phần tử có hai lựa chọn

Mỗi `nums[i]` có đúng **hai** giá trị có thể: `nums[i] + k` và `nums[i] - k` (khi `k = 0` hai giá trị trùng nhau). Có `2^n` cách chọn: với `n = 10^4` là không thể liệt kê.

In [2]:
nums, k = [1, 3, 6], 3
for signs in product((+1, -1), repeat=len(nums)):
    new = [x + s * k for x, s in zip(nums, signs)]
    print(signs, new, "điểm", max(new) - min(new))

(1, 1, 1) [4, 6, 9] điểm 5
(1, 1, -1) [4, 6, 3] điểm 3
(1, -1, 1) [4, 0, 9] điểm 9
(1, -1, -1) [4, 0, 3] điểm 4
(-1, 1, 1) [-2, 6, 9] điểm 11
(-1, 1, -1) [-2, 6, 3] điểm 8
(-1, -1, 1) [-2, 0, 9] điểm 11
(-1, -1, -1) [-2, 0, 3] điểm 5


### 3.2. Sắp xếp rồi cắt thành tiền tố và hậu tố

Sau khi sắp xếp `nums` tăng dần, một **điểm cắt** `i` (`0 <= i <= n`) nghĩa là: các phần tử `nums[0..i-1]` được `+k` và các phần tử `nums[i..n-1]` được `-k`. Có `n + 1` điểm cắt; `i = 0` (tất cả `-k`) và `i = n` (tất cả `+k`) đều cho điểm bằng `nums[-1] - nums[0]` (dịch cả dãy).

In [3]:
a = sorted([6, 1, 3])
k = 3
for i in range(len(a) + 1):
    new = [x + k for x in a[:i]] + [x - k for x in a[i:]]
    print(f"cắt tại i={i}: {new} -> điểm {max(new) - min(new)}")

cắt tại i=0: [-2, 0, 3] -> điểm 5
cắt tại i=1: [4, 0, 3] -> điểm 4
cắt tại i=2: [4, 6, 3] -> điểm 3
cắt tại i=3: [4, 6, 9] -> điểm 5


### 3.3. Công thức O(1) cho một điểm cắt

Với điểm cắt `i` (`1 <= i <= n - 1`): phần `+k` gồm `a[0..i-1]` có nhỏ nhất `a[0] + k` và lớn nhất `a[i-1] + k`; phần `-k` gồm `a[i..n-1]` có nhỏ nhất `a[i] - k` và lớn nhất `a[n-1] - k`. Vì vậy:

- nhỏ nhất toàn cục: `lo = min(a[0] + k, a[i] - k)`;
- lớn nhất toàn cục: `hi = max(a[i - 1] + k, a[n - 1] - k)`;
- điểm: `hi - lo`.

In [4]:
a, k = [1, 3, 6], 3
n = len(a)
for i in range(1, n):
    lo = min(a[0] + k, a[i] - k)
    hi = max(a[i - 1] + k, a[n - 1] - k)
    new = [x + k for x in a[:i]] + [x - k for x in a[i:]]
    print(f"i={i}: công thức {hi - lo}, tính trực tiếp {max(new) - min(new)}")
    assert hi - lo == max(new) - min(new)

i=1: công thức 4, tính trực tiếp 4
i=2: công thức 3, tính trực tiếp 3


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 3: `nums = [1, 3, 6]`, `k = 3`, đã sắp xếp. Khoảng cách ban đầu `6 - 1 = 5`.

| điểm cắt `i` | các giá trị sau đổi | điểm |
|---|---|---|
| `0` hoặc `3` (cùng dấu) | `[-2, 0, 3]` hoặc `[4, 6, 9]` | `5` |
| `1` (`+k` cho `1`, `-k` cho `3, 6`) | `[4, 0, 3]` | `4` |
| `2` (`+k` cho `1, 3`, `-k` cho `6`) | `[4, 6, 3]` | `3` |

Điểm nhỏ nhất là `3`. Trực giác: các số nhỏ **tăng lên**, các số lớn **giảm xuống** để chúng gặp nhau ở giữa; nhưng nếu `k` quá lớn thì cả hai nhóm "nhảy qua" nhau và không còn lợi: khi đó cách không cắt (cùng dấu) có thể tốt nhất.

## 5. Cách 1 — Brute Force: thử mọi tổ hợp dấu

**Ý tưởng.** Liệt kê mọi `2^n` cách chọn dấu `+k/-k` cho từng phần tử, tính `max - min` cho mỗi cách và lấy nhỏ nhất. Đây là định nghĩa trực tiếp nên cũng là **oracle**.

In [5]:
def smallest_range_brute(nums: List[int], k: int) -> int:
    best = None
    for signs in product((k, -k), repeat=len(nums)):
        new = [x + d for x, d in zip(nums, signs)]
        score = max(new) - min(new)
        if best is None or score < best:
            best = score
    return best

for nums, k, expected in examples:
    got = smallest_range_brute(nums, k)
    print(nums, k, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[1] 0 -> 0 OK
[0, 10] 2 -> 6 OK
[1, 3, 6] 3 -> 3 OK


**Độ phức tạp và nút thắt.** `O(2^n * n)`: với `n = 20` đã `~ 2 * 10^7` thao tác. Hãy xem thời gian tăng:

In [6]:
rng = random.Random(910)
for n in (8, 10, 12, 14, 16):
    nums = [rng.randint(0, 100) for _ in range(n)]
    t0 = time.perf_counter()
    smallest_range_brute(nums, 7)
    print(f"n={n:2}: {1000 * (time.perf_counter() - t0):9.1f} ms")

n= 8:       0.4 ms
n=10:       1.7 ms
n=12:       7.6 ms
n=14:      32.8 ms
n=16:      85.7 ms


Thời gian tăng gấp đôi mỗi phần tử. Nút thắt: ta xét cả những tổ hợp dấu **vô lý** (số lớn `+k` trong khi số nhỏ `-k`), trong khi chiến lược hợp lý luôn là "số nhỏ `+k`, số lớn `-k`".

## 6. Ý tưởng thuật toán / bước đột phá

**Bổ đề hoán đổi.** Giả sử trong một cách chọn có hai chỉ số `x < y` (theo thứ tự sắp xếp, `a[x] <= a[y]`) mà `a[x]` bị `-k` còn `a[y]` được `+k`. Hãy **đổi dấu cả hai**: `a[x] + k` và `a[y] - k`. Khoảng `[a[x] - k, a[y] + k]` (hai giá trị cũ) **chứa** khoảng giữa hai giá trị mới (vì `a[x] - k <= a[x] + k` và `a[y] - k <= a[y] + k`, đồng thời `a[x] - k <= a[y] - k` và `a[x] + k <= a[y] + k`), nên cả hai giá trị mới nằm trong đoạn `[a[x] - k, a[y] + k]`. Max/min toàn cục của mảng **không tăng/giảm** theo hướng xấu hơn, tức điểm **không tăng**. Lặp lại phép đổi tới khi không còn cặp như vậy, ta được cách tối ưu trong đó mọi số `+k` đứng **trước** mọi số `-k`: một **tiền tố** `+k` và một **hậu tố** `-k`.

Vậy chỉ cần xét `n + 1` điểm cắt.

| Cách | Ý tưởng |
|---|---|
| 2 | Sắp xếp, thử mọi điểm cắt, tính max/min bằng **quét** (`O(n)` mỗi điểm) |
| 3 | Sắp xếp, thử mọi điểm cắt bằng **công thức `O(1)`** (nộp) |
| 4 | Không dựa vào bổ đề: thử mọi **cận dưới** `L` của kết quả, mỗi phần tử chọn giá trị nhỏ nhất `>= L` |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Thử mọi điểm cắt, tính max/min bằng quét

**Giải thích từng đoạn.**

- `a = sorted(nums)`.
- Với mỗi `i` từ `0` đến `n`: dựng mảng mới `[x + k for x in a[:i]] + [x - k for x in a[i:]]`, tính `max - min`.
- Lấy nhỏ nhất.

Mỗi điểm cắt tốn `O(n)` nên tổng `O(n^2)`.

In [7]:
def smallest_range_cuts_scan(nums: List[int], k: int) -> int:
    a = sorted(nums)
    best = None
    for i in range(len(a) + 1):
        new = [x + k for x in a[:i]] + [x - k for x in a[i:]]
        score = max(new) - min(new)
        if best is None or score < best:
            best = score
    return best

for nums, k, expected in examples:
    got = smallest_range_cuts_scan(nums, k)
    print(nums, k, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[1] 0 -> 0 OK
[0, 10] 2 -> 6 OK
[1, 3, 6] 3 -> 3 OK


**Code nộp LeetCode (Cách 2):**

In [8]:
class Solution:
    def smallestRangeII(self, nums: List[int], k: int) -> int:
        a = sorted(nums)
        best = a[-1] - a[0]
        for i in range(1, len(a)):
            new = [x + k for x in a[:i]] + [x - k for x in a[i:]]
            best = min(best, max(new) - min(new))
        return best

sol_scan = Solution().smallestRangeII
assert [sol_scan(list(n), k) for n, k, _ in examples] == [e for _, _, e in examples]
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** `O(n^2)` thời gian (mỗi điểm cắt `O(n)`), `O(n)` bộ nhớ. Với `n = 10^4` là `10^8`: quá chậm trong Python.

---

### 7.2. Cách 3 — Thử mọi điểm cắt bằng công thức `O(1)` (cách nộp chính)

**Ý tưởng khác.** Dùng công thức ở mục 3.3: với điểm cắt `i`, nhỏ nhất `lo = min(a[0] + k, a[i] - k)`, lớn nhất `hi = max(a[i - 1] + k, a[-1] - k)`. Khởi tạo đáp án bằng `a[-1] - a[0]` (điểm cắt `0` hoặc `n`: cùng dấu), rồi thử `i = 1..n - 1`.

**Giải thích từng đoạn.**

- `ans = a[-1] - a[0]`: trường hợp mọi phần tử cùng dấu.
- `for i in range(1, n)`: mỗi `i` là một điểm cắt thật sự (cả hai phần đều không rỗng).
- `ans = min(ans, hi - lo)`.

In [9]:
def smallest_range_cuts(nums: List[int], k: int) -> int:
    a = sorted(nums)
    ans = a[-1] - a[0]
    for i in range(1, len(a)):
        lo = min(a[0] + k, a[i] - k)
        hi = max(a[i - 1] + k, a[-1] - k)
        ans = min(ans, hi - lo)
    return ans

for nums, k, expected in examples:
    got = smallest_range_cuts(nums, k)
    print(nums, k, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[1] 0 -> 0 OK
[0, 10] 2 -> 6 OK
[1, 3, 6] 3 -> 3 OK


**Code nộp LeetCode (Cách 3):**

In [10]:
class Solution:
    def smallestRangeII(self, nums: List[int], k: int) -> int:
        nums.sort()
        ans = nums[-1] - nums[0]
        for i in range(1, len(nums)):
            lo = min(nums[0] + k, nums[i] - k)
            hi = max(nums[i - 1] + k, nums[-1] - k)
            ans = min(ans, hi - lo)
        return ans

sol_cuts = Solution().smallestRangeII
assert [sol_cuts(list(n), k) for n, k, _ in examples] == [e for _, _, e in examples]
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** `O(n log n)` thời gian (sắp xếp; vòng lặp `O(n)`), `O(1)` hoặc `O(log n)` bộ nhớ phụ (sắp xếp tại chỗ).

---

### 7.3. Cách 4 — Liệt kê cận dưới của kết quả (không dùng bổ đề)

**Ý tưởng khác.** Giả sử giá trị nhỏ nhất sau khi đổi là `L`. Khi đó **mỗi** phần tử phải chọn một giá trị `>= L`; để làm lớn nhất nhỏ nhất ta cho nó chọn giá trị **nhỏ nhất trong hai (`x - k`, `x + k`) mà `>= L`** (nếu cả hai `< L` thì `L` không khả thi). Điểm khi đó `<= max(các lựa chọn) - L`. Vì giá trị nhỏ nhất thực sự của một cách tối ưu là một trong các giá trị `x +- k`, ta thử `L` trong tập `{x + k, x - k}` và lấy nhỏ nhất của `max(các lựa chọn) - L`.

**Giải thích từng đoạn.**

- `candidates = {x - k for x in nums} | {x + k for x in nums}`.
- Với mỗi `L`: với mỗi `x`, `options = [v for v in (x - k, x + k) if v >= L]`; nếu rỗng thì bỏ `L`; nếu không thì chọn `min(options)` và cập nhật `hi`.
- `ans = min(ans, hi - L)`.

Cách này **không dựa vào cấu trúc "tiền tố/hậu tố"**, nên là một kiểm chứng độc lập (nhưng `O(n^2)`).

In [11]:
def smallest_range_lower_bound(nums: List[int], k: int) -> int:
    candidates = {x - k for x in nums} | {x + k for x in nums}
    best = None
    for L in candidates:
        hi = None
        ok = True
        for x in nums:
            options = [v for v in (x - k, x + k) if v >= L]
            if not options:
                ok = False
                break
            v = min(options)
            hi = v if hi is None else max(hi, v)
        if ok:
            score = hi - L
            if best is None or score < best:
                best = score
    return best

for nums, k, expected in examples:
    got = smallest_range_lower_bound(nums, k)
    print(nums, k, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[1] 0 -> 0 OK
[0, 10] 2 -> 6 OK
[1, 3, 6] 3 -> 3 OK


**Code nộp LeetCode (Cách 4):** đúng nhưng chậm với `n = 10^4` (`O(n^2)`); dùng để kiểm chứng.

In [12]:
class Solution:
    def smallestRangeII(self, nums: List[int], k: int) -> int:
        candidates = {x - k for x in nums} | {x + k for x in nums}
        best = None
        for L in candidates:
            hi, ok = None, True
            for x in nums:
                options = [v for v in (x - k, x + k) if v >= L]
                if not options:
                    ok = False
                    break
                v = min(options)
                hi = v if hi is None else max(hi, v)
            if ok and (best is None or hi - L < best):
                best = hi - L
        return best

sol_lower = Solution().smallestRangeII
assert [sol_lower(list(n), k) for n, k, _ in examples] == [e for _, _, e in examples]
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** `O(n^2)` thời gian (`2n` ứng viên x `n` phần tử), `O(n)` bộ nhớ.

## 8. TRACE từng bước

### 8.1. Cách 3 cho ví dụ 3: `nums = [1, 3, 6]`, `k = 3`

In [13]:
def trace_cuts(nums, k):
    a = sorted(nums)
    ans = a[-1] - a[0]
    print(f"a = {a}, k = {k}, khởi tạo ans = a[-1] - a[0] = {ans}")
    for i in range(1, len(a)):
        lo = min(a[0] + k, a[i] - k)
        hi = max(a[i - 1] + k, a[-1] - k)
        new = [x + k for x in a[:i]] + [x - k for x in a[i:]]
        print(f"  cắt i={i}: +k cho {a[:i]}, -k cho {a[i:]}  -> {new}; lo=min({a[0] + k}, {a[i] - k})={lo}, hi=max({a[i - 1] + k}, {a[-1] - k})={hi}, điểm {hi - lo}")
        ans = min(ans, hi - lo)
    return ans

assert trace_cuts([6, 1, 3], 3) == 3

a = [1, 3, 6], k = 3, khởi tạo ans = a[-1] - a[0] = 5
  cắt i=1: +k cho [1], -k cho [3, 6]  -> [4, 0, 3]; lo=min(4, 0)=0, hi=max(4, 3)=4, điểm 4
  cắt i=2: +k cho [1, 3], -k cho [6]  -> [4, 6, 3]; lo=min(4, 3)=3, hi=max(6, 3)=6, điểm 3


### 8.2. Trường hợp `k` quá lớn: `nums = [0, 10]`, `k = 20`

Cắt `i = 1`: `lo = min(0 + 20, 10 - 20) = -10`, `hi = max(0 + 20, 10 - 20) = 20`, điểm `30` (rất xấu); cùng dấu cho điểm `10`. Đáp án là `10` nhờ khởi tạo `ans = a[-1] - a[0]`.

In [14]:
assert trace_cuts([0, 10], 20) == 10

a = [0, 10], k = 20, khởi tạo ans = a[-1] - a[0] = 10
  cắt i=1: +k cho [0], -k cho [10]  -> [20, -10]; lo=min(20, -10)=-10, hi=max(20, -10)=20, điểm 30


### 8.3. Cách 4 (cận dưới) cho `nums = [1, 3, 6]`, `k = 3`

Các ứng viên `L` thuộc `{-2, 0, 3, 4, 6, 9}`. Với `L = 3`: `1 -> 4` (vì `1 - 3 = -2 < 3`), `3 -> 6` (`0 < 3`), `6 -> 3` (`3 >= 3` chọn giá trị nhỏ nhất `>= 3`): `hi = 6`, điểm `3`.

In [15]:
nums, k = [1, 3, 6], 3
for L in sorted({x - k for x in nums} | {x + k for x in nums}):
    picks = []
    for x in nums:
        options = [v for v in (x - k, x + k) if v >= L]
        picks.append(min(options) if options else None)
    if None in picks:
        print(f"L={L:3}: không khả thi (phần tử {nums[picks.index(None)]} không có lựa chọn >= L)")
    else:
        print(f"L={L:3}: chọn {picks} -> điểm <= {max(picks) - L}")

L= -2: chọn [-2, 0, 3] -> điểm <= 5
L=  0: chọn [4, 0, 3] -> điểm <= 4
L=  3: chọn [4, 6, 3] -> điểm <= 3
L=  4: chọn [4, 6, 9] -> điểm <= 5
L=  6: không khả thi (phần tử 1 không có lựa chọn >= L)
L=  9: không khả thi (phần tử 1 không có lựa chọn >= L)


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (sắp xếp + thử mọi điểm cắt bằng công thức `O(1)`): `O(n log n)`, ngắn gọn.

```python
class Solution:
    def smallestRangeII(self, nums: List[int], k: int) -> int:
        nums.sort()
        ans = nums[-1] - nums[0]
        for i in range(1, len(nums)):
            lo = min(nums[0] + k, nums[i] - k)
            hi = max(nums[i - 1] + k, nums[-1] - k)
            ans = min(ans, hi - lo)
        return ans
```

In [16]:
class Solution:
    def smallestRangeII(self, nums: List[int], k: int) -> int:
        nums.sort()
        ans = nums[-1] - nums[0]
        for i in range(1, len(nums)):
            lo = min(nums[0] + k, nums[i] - k)
            hi = max(nums[i - 1] + k, nums[-1] - k)
            ans = min(ans, hi - lo)
        return ans

sol = Solution()
for nums, k, expected in examples:
    got = sol.smallestRangeII(list(nums), k)
    print(nums, k, "->", got, "OK" if got == expected else "SAI")
    assert got == expected

[1] 0 -> 0 OK
[0, 10] 2 -> 6 OK
[1, 3, 6] 3 -> 3 OK


## 10. Vì sao đúng

**(a) Bổ đề hoán đổi.** Sắp xếp `a[0] <= a[1] <= ... <= a[n-1]`. Xét cách chọn dấu bất kỳ và hai chỉ số `x < y` với `a[x]` nhận `-k` và `a[y]` nhận `+k`. Gọi giá trị cũ là `p = a[x] - k` và `q = a[y] + k`; giá trị mới (đổi dấu cả hai) là `p' = a[x] + k` và `q' = a[y] - k`. Ta có `p <= p'` và `q' <= q`, nên cả `p'` và `q'` nằm trong `[p, q]`? Cần `p <= q'` và `p' <= q`: `p = a[x] - k <= a[y] - k = q'` (vì `a[x] <= a[y]`), và `p' = a[x] + k <= a[y] + k = q`. Do đó `[p', q']` hoặc `[q', p']` đều nằm trong `[p, q]`. Vì mọi giá trị khác giữ nguyên, **max không tăng và min không giảm**: điểm không tăng.

**(b) Cấu trúc tiền tố/hậu tố.** Mỗi lần đổi như (a) làm giảm số cặp "nghịch" (`-k` ở trước `+k`) nên sau hữu hạn bước không còn cặp nào, tức là mọi `+k` đứng trước mọi `-k`. Vậy tồn tại cách tối ưu ứng với một điểm cắt `i`.

**(c) Công thức điểm cắt.** Với điểm cắt `1 <= i <= n - 1`: phần `+k` có min `a[0] + k` và max `a[i-1] + k`; phần `-k` có min `a[i] - k` và max `a[n-1] - k`. Toàn cục: `lo = min(a[0] + k, a[i] - k)`, `hi = max(a[i-1] + k, a[n-1] - k)`. Điểm cắt `i = 0` hoặc `i = n` (cùng dấu) cho `a[n-1] - a[0]`; với `n = 1` cũng là `0`.

**(d) Cách 4.** Nếu giá trị nhỏ nhất của một cách tối ưu là `L`, mọi phần tử chọn giá trị `>= L`; chọn giá trị **nhỏ nhất `>= L`** của mỗi phần tử cho max nhỏ nhất. `L` phải là một trong `x +- k`. Do đó `min over L` của `max - L` bằng đáp án (mọi cách chọn hợp lệ cho `max - L <=` điểm thực; và với `L` đúng bằng min của cách tối ưu thì bằng).

**Kiểm chứng bằng máy.** Oracle **vét cạn `2^n`** (`n <= 14`) so với mọi lời giải; với `n` lớn dùng thêm một cách cài đặt `O(n)` khác hẳn: **mảng min/max tiền tố và hậu tố** sau khi sắp xếp (không dùng công thức `min/max` bốn số), và Cách 4 (không dựa vào bổ đề) cho `n <= 300`.

In [17]:
def oracle_prefix_suffix(nums: List[int], k: int) -> int:
    """O(n): sau khi sắp xếp, với mỗi điểm cắt i lấy min/max của phần +k và phần -k bằng mảng tiền tố/hậu tố."""
    a = sorted(nums)
    n = len(a)
    plus_min = [a[0] + k] * n                         # plus_min[i] = min của a[0..i] + k
    plus_max = [a[i] + k for i in range(n)]           # plus_max[i] = max của a[0..i] + k (tăng dần)
    minus_min = [a[i] - k for i in range(n)]          # minus_min[i] = min của a[i..n-1] - k (a tăng nên là a[i] - k)
    minus_max = [a[-1] - k] * n
    best = a[-1] - a[0]
    for i in range(1, n):                              # +k cho a[0..i-1], -k cho a[i..]
        lo = min(plus_min[i - 1], minus_min[i])
        hi = max(plus_max[i - 1], minus_max[i])
        best = min(best, hi - lo)
    return best

rng = random.Random(1910)
for _ in range(400):
    n = rng.randint(1, 10)
    nums = [rng.randint(0, 30) for _ in range(n)]
    k = rng.randint(0, 20)
    b = smallest_range_brute(nums, k)
    assert oracle_prefix_suffix(nums, k) == b, (nums, k)
    assert smallest_range_lower_bound(nums, k) == b, (nums, k)
print("vét cạn 2^n, tiền tố/hậu tố và cận dưới khớp nhau trên 400 ca ngẫu nhiên")

vét cạn 2^n, tiền tố/hậu tố và cận dưới khớp nhau trên 400 ca ngẫu nhiên


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mọi tổ hợp dấu | `O(2^n * n)` | `O(n)` |
| 2. Mọi điểm cắt, quét | `O(n^2)` | `O(n)` |
| 3. Mọi điểm cắt, `O(1)` | `O(n log n)` | `O(1)` phụ |
| 4. Cận dưới | `O(n^2)` | `O(n)` |

Đo thử:

In [18]:
rng = random.Random(11)
for n in (200, 1000, 3000):
    nums = [rng.randint(0, 10 ** 4) for _ in range(n)]
    k = rng.randint(0, 10 ** 4)
    row = []
    for name, fn in (("Cách 2", sol_scan), ("Cách 3", sol_cuts), ("Cách 4", sol_lower)):
        if name == "Cách 4" and n > 1000:
            continue
        t0 = time.perf_counter()
        got = fn(list(nums), k)
        row.append(f"{name}: {1000 * (time.perf_counter() - t0):9.2f} ms")
        assert got == oracle_prefix_suffix(nums, k), (name, n)
    print(f"n={n:5}: " + " | ".join(row))
nums = [rng.randint(0, 10 ** 4) for _ in range(10 ** 4)]
t0 = time.perf_counter()
r = sol_cuts(list(nums), 4321)
print(f"Cách 3, n = 10^4 (cỡ tối đa): {1000 * (time.perf_counter() - t0):.2f} ms, đáp án {r}")
assert r == oracle_prefix_suffix(nums, 4321)

n=  200: Cách 2:      4.30 ms | Cách 3:      0.10 ms | Cách 4:      4.53 ms


n= 1000: Cách 2:     94.46 ms | Cách 3:      0.55 ms | Cách 4:    404.87 ms


n= 3000: Cách 2:    660.24 ms | Cách 3:      1.68 ms
Cách 3, n = 10^4 (cỡ tối đa): 6.66 ms, đáp án 8633


## 12. Bộ test

Gồm: ví dụ của đề, edge case, phản ví dụ cho các cách sai (tìm bằng tìm kiếm có seed), đối chiếu mọi cách với oracle vét cạn `2^n`, test ngẫu nhiên có seed (đúng ràng buộc) và tính chất.

In [19]:
def solutions_list():
    return {
        "cuts_scan": smallest_range_cuts_scan,
        "cuts": smallest_range_cuts,
        "lower": smallest_range_lower_bound,
        "prefix_suffix": oracle_prefix_suffix,
        "sol_scan": sol_scan,
        "sol_cuts": sol_cuts,
        "sol_lower": sol_lower,
        "final": Solution().smallestRangeII,
    }

def run_all(nums, k, expected=None, brute_limit=14):
    ref = smallest_range_brute(nums, k) if len(nums) <= brute_limit else oracle_prefix_suffix(nums, k)
    for name, fn in solutions_list().items():
        assert fn(list(nums), k) == ref, (name, nums, k)
    if expected is not None:
        assert ref == expected, ("kỳ vọng sai", nums, k, expected, ref)
    return ref

# 12.1 ví dụ đề
for nums, k, expected in examples:
    assert run_all(nums, k, expected) == expected
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ([5], 7, 0),                          # một phần tử: điểm 0
    ([0, 0], 5, 0),                       # hai phần tử bằng nhau, cùng dấu
    ([0, 10], 0, 10),                     # k = 0: không đổi, điểm = max - min
    ([1, 3, 6], 0, 5),                    # k = 0
    ([0, 1], 5, 1),                       # k lớn: cùng dấu tốt nhất (cắt chéo cho 9)
    ([0, 10], 2, 6),                      # ví dụ 2
    ([0, 10], 5, 0),                      # +5 cho 0 và -5 cho 10: cả hai thành 5
    ([0, 10], 20, 10),                    # k quá lớn: cùng dấu
    ([2, 2, 2], 3, 0),                    # toàn bằng nhau
    ([1, 5, 8, 10], 3, 4),                # [4, 8, 5, 7]: điểm 4
    ([0, 10000], 10000, 10000),           # biên: cùng dấu cho 10000; chéo cho 10000 hoặc 30000
    ([0, 4, 7, 10], 5, 6),                # +5 cho 0; -5 cho 4, 7, 10: [5, -1, 2, 5] -> 6
]
for nums, k, expected in edge:
    run_all(nums, k, expected)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 12 ca


In [20]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm có seed rồi xác nhận lời giải đúng không bị lỗi)
def wrong_no_init(nums, k):                       # bỏ trường hợp "không cắt" (cùng dấu)
    a = sorted(nums)
    ans = float("inf")
    for i in range(1, len(a)):
        lo = min(a[0] + k, a[i] - k)
        hi = max(a[i - 1] + k, a[-1] - k)
        ans = min(ans, hi - lo)
    return ans if ans != float("inf") else 0

def wrong_not_sorted(nums, k):                    # quên sắp xếp
    ans = max(nums) - min(nums)
    for i in range(1, len(nums)):
        lo = min(nums[0] + k, nums[i] - k)
        hi = max(nums[i - 1] + k, nums[-1] - k)
        ans = min(ans, hi - lo)
    return ans

def wrong_simple_formula(nums, k):                # mi = a[0] + k, mx = a[-1] - k không có min/max
    a = sorted(nums)
    ans = a[-1] - a[0]
    for i in range(1, len(a)):
        ans = min(ans, abs((a[i - 1] + k) - (a[i] - k)))
    return ans

def wrong_greedy_mean(nums, k):                   # tham lam từng phần tử theo trung bình
    mean = sum(nums) / len(nums)
    new = [x + k if x < mean else x - k for x in nums]
    return max(new) - min(new)

def wrong_range_one(nums, k):                     # nhầm với Smallest Range I (908): cộng tuỳ ý trong [-k, k]
    return max(0, max(nums) - min(nums) - 2 * k)

def find_counter(wrong, tries=20000, seed=7):
    rng = random.Random(seed)
    for _ in range(tries):
        n = rng.randint(1, 6)
        nums = [rng.randint(0, 12) for _ in range(n)]
        k = rng.randint(0, 8)
        if wrong(nums, k) != smallest_range_brute(nums, k):
            return nums, k
    return None

counters = {}
for name, fn in (("bỏ trường hợp không cắt", wrong_no_init), ("quên sắp xếp", wrong_not_sorted),
                 ("công thức bỏ min/max", wrong_simple_formula), ("tham lam theo trung bình", wrong_greedy_mean),
                 ("nhầm với Smallest Range I", wrong_range_one)):
    c = find_counter(fn)
    assert c is not None, name
    counters[name] = c
    nums, k = c
    print(f"{name:28} nums={nums} k={k}: sai -> {fn(nums, k)}, đúng -> {smallest_range_brute(nums, k)}")
    run_all(nums, k)
print("12.3 phản ví dụ: OK")

bỏ trường hợp không cắt      nums=[9, 6, 0, 3, 0] k=8: sai -> 13, đúng -> 9
quên sắp xếp                 nums=[9, 0, 8] k=3: sai -> 0, đúng -> 3
công thức bỏ min/max         nums=[2, 6, 10] k=0: sai -> 4, đúng -> 8
tham lam theo trung bình     nums=[9, 6, 0, 3, 0] k=8: sai -> 13, đúng -> 9
nhầm với Smallest Range I    nums=[1, 3, 1, 8] k=6: sai -> 0, đúng -> 7
12.3 phản ví dụ: OK


In [21]:
# 12.4 ngẫu nhiên nhỏ: so mọi cách với oracle vét cạn (n <= 12)
rng = random.Random(4910)
for _ in range(1500):
    n = rng.randint(1, 12)
    vmax = rng.choice([3, 10, 100, 10 ** 4])
    nums = [rng.randint(0, vmax) for _ in range(n)]
    k = rng.randint(0, rng.choice([2, 10, 10 ** 4]))
    run_all(nums, k)
print("12.4 ngẫu nhiên n <= 12 (1500 ca): OK")

# 12.5 trung bình: n tới 300, so với oracle tiền tố/hậu tố và cận dưới (không dựa vào bổ đề)
rng = random.Random(5910)
for _ in range(200):
    n = rng.randint(13, 300)
    vmax = rng.choice([10, 1000, 10 ** 4])
    nums = [rng.randint(0, vmax) for _ in range(n)]
    k = rng.randint(0, rng.choice([5, 1000, 10 ** 4]))
    run_all(nums, k, brute_limit=0)
print("12.5 ngẫu nhiên n tới 300 (200 ca): OK")

# 12.6 lớn n = 10^4 (đúng ràng buộc), so Cách 3 với oracle tiền tố/hậu tố và Cách 2 trên mẫu con
rng = random.Random(6910)
for vmax, kmax in ((10 ** 4, 10 ** 4), (100, 50), (10 ** 4, 0), (5, 10 ** 4)):
    nums = [rng.randint(0, vmax) for _ in range(10 ** 4)]
    k = rng.randint(0, kmax)
    assert Solution().smallestRangeII(list(nums), k) == oracle_prefix_suffix(nums, k) == sol_cuts(list(nums), k)
print("12.6 n = 10^4: OK")

# 12.7 tính chất: không phụ thuộc thứ tự; k tăng thì đáp án không vượt quá max - min; tịnh tiến nums không đổi đáp án
rng = random.Random(7910)
for _ in range(300):
    n = rng.randint(1, 12)
    nums = [rng.randint(0, 50) for _ in range(n)]
    k = rng.randint(0, 30)
    base = Solution().smallestRangeII(list(nums), k)
    shuffled = nums[:]
    rng.shuffle(shuffled)
    assert Solution().smallestRangeII(shuffled, k) == base
    assert base <= max(nums) - min(nums)
    shift = rng.randint(0, 100)
    assert Solution().smallestRangeII([x + shift for x in nums], k) == base
print("12.7 tính chất (hoán vị, <= max-min, tịnh tiến): OK")

12.4 ngẫu nhiên n <= 12 (1500 ca): OK


12.5 ngẫu nhiên n tới 300 (200 ca): OK
12.6 n = 10^4: OK
12.7 tính chất (hoán vị, <= max-min, tịnh tiến): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Bỏ trường hợp "không cắt" (mọi phần tử cùng dấu)

Nếu `k` lớn thì chia đôi dãy làm khoảng cách rộng thêm; cách tốt nhất là cho **mọi** phần tử cùng dấu (điểm `max - min` ban đầu).

In [22]:
nums, k = counters["bỏ trường hợp không cắt"]
print("nums =", nums, "k =", k, "| sai ->", wrong_no_init(nums, k), "| đúng ->", Solution().smallestRangeII(list(nums), k))
assert wrong_no_init(nums, k) != Solution().smallestRangeII(list(nums), k)
print("ví dụ gọn: [0, 1], k = 5 ->", wrong_no_init([0, 1], 5), "(sai) so với", Solution().smallestRangeII([0, 1], 5))

nums = [9, 6, 0, 3, 0] k = 8 | sai -> 13 | đúng -> 9
ví dụ gọn: [0, 1], k = 5 -> 9 (sai) so với 1


Vì sao sai: mọi điểm cắt thật sự cho điểm lớn hơn `max - min` khi `k` lớn. **Đúng:** khởi tạo `ans = nums[-1] - nums[0]`.

### Lỗi 2 — Quên sắp xếp

Công thức điểm cắt chỉ đúng khi `nums` đã sắp xếp (tiền tố = các số nhỏ nhất).

In [23]:
nums, k = counters["quên sắp xếp"]
print("nums =", nums, "k =", k, "| sai ->", wrong_not_sorted(nums, k), "| đúng ->", Solution().smallestRangeII(list(nums), k))
assert wrong_not_sorted(nums, k) != Solution().smallestRangeII(list(nums), k)

nums = [9, 0, 8] k = 3 | sai -> 0 | đúng -> 3


Vì sao sai: chia "tiền tố" theo thứ tự ban đầu là chia ngẫu nhiên, không phải "số nhỏ `+k`, số lớn `-k`". **Đúng:** `nums.sort()` trước.

### Lỗi 3 — Công thức bỏ `min`/`max`

`lo = a[0] + k` và `hi = a[-1] - k` (hoặc lấy trị tuyệt đối của hiệu hai đầu cắt) bỏ qua trường hợp hai nhóm "nhảy qua" nhau khi `k` lớn.

In [24]:
nums, k = counters["công thức bỏ min/max"]
print("nums =", nums, "k =", k, "| sai ->", wrong_simple_formula(nums, k), "| đúng ->", Solution().smallestRangeII(list(nums), k))
assert wrong_simple_formula(nums, k) != Solution().smallestRangeII(list(nums), k)

nums = [2, 6, 10] k = 0 | sai -> 4 | đúng -> 8


Vì sao sai: khi `a[i-1] + k` lớn hơn `a[-1] - k` hoặc `a[0] + k` lớn hơn `a[i] - k`, các cực trị là các giá trị khác. **Đúng:** `lo = min(a[0] + k, a[i] - k)`, `hi = max(a[i-1] + k, a[-1] - k)`.

### Lỗi 4 — Tham lam từng phần tử

Quyết định dấu của mỗi phần tử độc lập (ví dụ theo trung bình) bỏ qua ràng buộc toàn cục `max - min`.

In [25]:
nums, k = counters["tham lam theo trung bình"]
print("nums =", nums, "k =", k, "| sai ->", wrong_greedy_mean(nums, k), "| đúng ->", Solution().smallestRangeII(list(nums), k))
assert wrong_greedy_mean(nums, k) != Solution().smallestRangeII(list(nums), k)

nums = [9, 6, 0, 3, 0] k = 8 | sai -> 13 | đúng -> 9


Vì sao sai: dấu tối ưu là một **điểm cắt** theo thứ tự sắp xếp, không nhất thiết trùng ngưỡng trung bình. **Đúng:** thử mọi điểm cắt.

### Lỗi 5 — Nhầm với Smallest Range I (908)

Ở 908 mỗi số được cộng **một số bất kỳ trong `[-k, k]`**, đáp án `max(0, max - min - 2k)`. Ở 910 phải cộng **đúng `+k` hoặc `-k`**.

In [26]:
nums, k = counters["nhầm với Smallest Range I"]
print("nums =", nums, "k =", k, "| công thức 908 ->", wrong_range_one(nums, k), "| đúng cho 910 ->", Solution().smallestRangeII(list(nums), k))
assert wrong_range_one(nums, k) != Solution().smallestRangeII(list(nums), k)
print("ví dụ 3: [1,3,6], k=3 ->", wrong_range_one([1, 3, 6], 3), "(908) so với", Solution().smallestRangeII([1, 3, 6], 3), "(910)")

nums = [1, 3, 1, 8] k = 6 | công thức 908 -> 0 | đúng cho 910 -> 7
ví dụ 3: [1,3,6], k=3 -> 0 (908) so với 3 (910)


Vì sao sai: bài 910 **không** cho phép giữ nguyên hoặc cộng giá trị trung gian, nên không thể thu hẹp tự do. **Đúng:** thử mọi điểm cắt theo dấu `+k/-k`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "mỗi phần tử có hai lựa chọn cố định, tối thiểu hóa `max - min`" ⇒ **sắp xếp** + **điểm cắt** (số nhỏ chọn một phía, số lớn chọn phía kia).
- **Lập luận hoán đổi** để chứng minh cấu trúc tiền tố/hậu tố.
- Luôn xét trường hợp **không cắt** (cùng dấu) khi `k` lớn.
- Công thức điểm cắt: `lo = min(a[0] + k, a[i] - k)`, `hi = max(a[i-1] + k, a[-1] - k)`.
- Cùng họ: Smallest Range I (908), Minimize Difference Between Highest and Lowest of K Scores, Minimum Difference Between Largest and Smallest Value in Three Moves (1509).
- Test nên có: `n = 1`, `k = 0`, `k` rất lớn, dãy toàn bằng nhau, và đối chiếu với vét cạn `2^n`.

## 15. Biến thể

1. **Điểm lớn nhất** (thay vì nhỏ nhất): cho mỗi phần tử lớn nhất `+k` và nhỏ nhất `-k`; đáp án `max - min + 2k` khi `n >= 2` (kiểm bằng vét cạn bên dưới).
2. **Smallest Range I (908):** mỗi phần tử cộng số tùy ý trong `[-k, k]`: đáp án `max(0, max - min - 2k)`.
3. **Trả về dấu `+/-`** cho từng chỉ số (bài tập 5).
4. Bài liên quan: Smallest Range Covering Elements from K Lists (632).

In [27]:
def largest_range_brute(nums, k):
    return max(max(new) - min(new) for new in ([x + d for x, d in zip(nums, s)] for s in product((k, -k), repeat=len(nums))))

def largest_range(nums, k):
    return max(nums) - min(nums) + 2 * k if len(nums) >= 2 else 0

def smallest_range_I_brute(nums, k):
    # mỗi phần tử cộng một số nguyên trong [-k, k]
    best = None
    for deltas in product(range(-k, k + 1), repeat=len(nums)):
        new = [x + d for x, d in zip(nums, deltas)]
        score = max(new) - min(new)
        best = score if best is None or score < best else best
    return best

rng = random.Random(15)
for _ in range(300):
    n = rng.randint(1, 6)
    nums = [rng.randint(0, 15) for _ in range(n)]
    k = rng.randint(0, 6)
    if n >= 2:
        assert largest_range(nums, k) == largest_range_brute(nums, k), (nums, k)
    if n <= 4 and k <= 3:
        assert smallest_range_I_brute(nums, k) == max(0, max(nums) - min(nums) - 2 * k), (nums, k)
print("biến thể 1 và 2 khớp vét cạn")

biến thể 1 và 2 khớp vét cạn


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi tổ hợp dấu | Định nghĩa | `O(2^n n)` | `O(n)` | Oracle, `n <= 14` |
| 2. Cắt + quét | Tiền tố `+k`, hậu tố `-k` | `O(n^2)` | `O(n)` | Dễ hiểu |
| 3. Cắt + `O(1)` | Công thức điểm cắt | `O(n log n)` | `O(1)` | **Cách nộp chính** |
| 4. Cận dưới | Không dùng bổ đề | `O(n^2)` | `O(n)` | Kiểm chứng độc lập |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output ba dòng, rồi chạy để kiểm tra:

```python
a, k = [1, 3, 6], 3
print(min(a[0] + k, a[1] - k), max(a[0] + k, a[-1] - k))
print(min(a[0] + k, a[2] - k), max(a[1] + k, a[-1] - k))
print(a[-1] - a[0])
```

In [28]:
# Bài 1: ghi dự đoán của bạn
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `nums = [1, 5, 8, 10]` và `k = 3`: ghi điểm tốt nhất và một cách chọn dấu (một danh sách `+1/-1` theo thứ tự đã sắp xếp).

In [29]:
# Bài 2: điền kết quả tính tay
hand_best = None
hand_signs = None             # ví dụ [+1, +1, -1, -1]

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành thử mọi điểm cắt bằng công thức `O(1)`.

In [30]:
def exercise_3_cuts(nums: List[int], k: int) -> int:
    a = sorted(nums)
    ans = a[-1] - a[0]
    for i in range(1, len(a)):
        # TODO 1: lo = nhỏ nhất toàn cục khi cắt tại i
        # TODO 2: hi = lớn nhất toàn cục khi cắt tại i
        # TODO 3: cập nhật ans
        pass
    return ans


# for nums, k, expected in examples:
#     assert exercise_3_cuts(nums, k) == expected

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 1, hãy tự viết `exercise_4_brute(nums, k)` thử mọi tổ hợp dấu bằng `itertools.product`.

In [31]:
def exercise_4_brute(nums: List[int], k: int) -> int:
    # TODO
    pass


# for nums, k, expected in examples:
#     assert exercise_4_brute(nums, k) == expected

### Bài 5 — Mini challenge: trả về các dấu tối ưu (khó)

Viết `exercise_5_signs(nums, k)` trả về một danh sách `signs` (mỗi phần tử `+1` hoặc `-1`, theo **thứ tự chỉ số gốc** của `nums`) sao cho `nums[i] + signs[i] * k` cho điểm **nhỏ nhất**. Có nhiều đáp án; ta kiểm tra bằng điểm.

```python
exercise_5_signs([1, 5, 8, 10], 3)  -> ví dụ [+1, +1, -1, -1]
```
Gợi ý: tìm điểm cắt tối ưu trên bản sắp xếp (nhớ chỉ số gốc), rồi gán `+1` cho các chỉ số gốc của tiền tố.

In [32]:
def exercise_5_signs(nums: List[int], k: int) -> List[int]:
    # TODO
    pass


# signs = exercise_5_signs([1, 5, 8, 10], 3)
# assert len(signs) == 4 and set(signs) <= {1, -1}
# new = [x + s * 3 for x, s in zip([1, 5, 8, 10], signs)]
# assert max(new) - min(new) == 4

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: với `i = 1`: nhóm `+k` là `{1}`, nhóm `-k` là `{3, 6}`.
- Mức 2: `i = 1`: `min(4, 0) = 0`, `max(4, 3) = 4`; `i = 2`: `min(4, 3) = 3`, `max(3 + 3, 3) = 6`.
- Mức 3: `0 4`, `3 6`, `5`.

**Bài 2**
- Mức 1: thử các điểm cắt `i = 1, 2, 3` và so với `10 - 1 = 9`.
- Mức 2: `i = 2`: `+k` cho `1, 5`, `-k` cho `8, 10`: các giá trị `4, 8, 5, 7`.
- Mức 3: điểm `4`, dấu `[+1, +1, -1, -1]`.

**Bài 3**
- Mức 1: `lo = min(a[0] + k, a[i] - k)`.
- Mức 2: `hi = max(a[i - 1] + k, a[-1] - k)`.
- Mức 3: `ans = min(ans, hi - lo)`.

**Bài 4**
- Mức 1: `for signs in product((k, -k), repeat=len(nums))`.
- Mức 2: `new = [x + d for x, d in zip(nums, signs)]`.
- Mức 3: giữ `min` của `max(new) - min(new)`.

**Bài 5**
- Mức 1: sắp xếp các cặp `(giá trị, chỉ số gốc)`.
- Mức 2: duyệt điểm cắt `i` (kể cả `i = n` nghĩa là cùng dấu) và chọn điểm cắt cho điểm nhỏ nhất.
- Mức 3: gán `signs[chỉ số gốc] = +1` cho `i` phần tử đầu tiên và `-1` cho phần còn lại.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [33]:
a, k = [1, 3, 6], 3
print(min(a[0] + k, a[1] - k), max(a[0] + k, a[-1] - k))
print(min(a[0] + k, a[2] - k), max(a[1] + k, a[-1] - k))
print(a[-1] - a[0])

assert (min(a[0] + k, a[1] - k), max(a[0] + k, a[-1] - k)) == (0, 4)
assert (min(a[0] + k, a[2] - k), max(a[1] + k, a[-1] - k)) == (3, 6)
assert a[-1] - a[0] == 5

0 4
3 6
5


### Đáp án Bài 2

Điểm cắt `i = 2` cho `[4, 8, 5, 7]`: điểm `4`.

In [34]:
nums, k = [1, 5, 8, 10], 3
hand_best_solution = 4
hand_signs_solution = [+1, +1, -1, -1]
assert Solution().smallestRangeII(list(nums), k) == hand_best_solution == smallest_range_brute(nums, k)
new = [x + s * k for x, s in zip(nums, hand_signs_solution)]
assert max(new) - min(new) == hand_best_solution
print("Bài 2 OK", new)

Bài 2 OK [4, 8, 5, 7]


### Đáp án Bài 3

In [35]:
def exercise_3_cuts_solution(nums: List[int], k: int) -> int:
    a = sorted(nums)
    ans = a[-1] - a[0]
    for i in range(1, len(a)):
        lo = min(a[0] + k, a[i] - k)
        hi = max(a[i - 1] + k, a[-1] - k)
        ans = min(ans, hi - lo)
    return ans

for nums, k, expected in examples:
    assert exercise_3_cuts_solution(nums, k) == expected
rng = random.Random(3)
for _ in range(300):
    nums = [rng.randint(0, 30) for _ in range(rng.randint(1, 10))]
    k = rng.randint(0, 20)
    assert exercise_3_cuts_solution(nums, k) == smallest_range_brute(nums, k)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [36]:
def exercise_4_brute_solution(nums: List[int], k: int) -> int:
    best = None
    for signs in product((k, -k), repeat=len(nums)):
        new = [x + d for x, d in zip(nums, signs)]
        score = max(new) - min(new)
        if best is None or score < best:
            best = score
    return best

for nums, k, expected in examples:
    assert exercise_4_brute_solution(nums, k) == expected
rng = random.Random(4)
for _ in range(200):
    nums = [rng.randint(0, 30) for _ in range(rng.randint(1, 9))]
    k = rng.randint(0, 20)
    assert exercise_4_brute_solution(nums, k) == Solution().smallestRangeII(list(nums), k)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [37]:
def exercise_5_signs_solution(nums: List[int], k: int) -> List[int]:
    order = sorted(range(len(nums)), key=lambda i: nums[i])
    a = [nums[i] for i in order]
    n = len(a)
    best_i, best_score = n, a[-1] - a[0]
    for i in range(1, n):
        lo = min(a[0] + k, a[i] - k)
        hi = max(a[i - 1] + k, a[-1] - k)
        if hi - lo < best_score:
            best_i, best_score = i, hi - lo
    signs = [0] * n
    for rank, idx in enumerate(order):
        signs[idx] = +1 if rank < best_i else -1
    return signs

def check_signs(nums, k, signs):
    return len(signs) == len(nums) and set(signs) <= {1, -1} and \
        (lambda new: max(new) - min(new))([x + s * k for x, s in zip(nums, signs)]) == smallest_range_brute(nums, k)

signs = exercise_5_signs_solution([1, 5, 8, 10], 3)
assert check_signs([1, 5, 8, 10], 3, signs)
rng = random.Random(5)
for _ in range(400):
    nums = [rng.randint(0, 30) for _ in range(rng.randint(1, 9))]
    k = rng.randint(0, 20)
    assert check_signs(nums, k, exercise_5_signs_solution(nums, k)), (nums, k)
print("Bài 5 OK", signs)

Bài 5 OK [1, 1, -1, -1]


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (sắp xếp + thử mọi điểm cắt)

```python
class Solution:
    def smallestRangeII(self, nums: List[int], k: int) -> int:
        nums.sort()
        ans = nums[-1] - nums[0]                         # trường hợp mọi phần tử cùng dấu
        for i in range(1, len(nums)):                    # nums[:i] được +k, nums[i:] được -k
            lo = min(nums[0] + k, nums[i] - k)
            hi = max(nums[i - 1] + k, nums[-1] - k)
            ans = min(ans, hi - lo)
        return ans
```

Công thức nhớ nhanh: số nhỏ `+k`, số lớn `-k`; điểm cắt `i`: `lo = min(a[0]+k, a[i]-k)`, `hi = max(a[i-1]+k, a[-1]-k)`; khởi tạo `ans = a[-1] - a[0]`.

### Checklist trước khi submit

- [ ] Đã **sắp xếp** `nums`?
- [ ] Khởi tạo `ans = nums[-1] - nums[0]` (không cắt)?
- [ ] `lo` và `hi` dùng `min`/`max` của bốn giá trị (không rút gọn sai)?
- [ ] Không nhầm với Smallest Range I (908)?
- [ ] Đã thử: `n = 1`, `k = 0`, `k` rất lớn, dãy toàn bằng nhau, `n = 10^4`?
- [ ] Tên method đúng `smallestRangeII(self, nums, k)` và trả `int`?